#### **1. Introduction: What is Text Splitting?**

*   **Definition:** Text splitting is the process of breaking down a large piece of text (like a PDF, article, or book) into smaller, manageable chunks.
*   **Why is it needed?** Processing a massive document all at once with an LLM is difficult and inefficient. Splitting it into smaller pieces makes it manageable.
*   **What is a Text Splitter?** A tool or component (like in LangChain) that performs this text-splitting operation.

#### **2. Why is Text Splitting Important in Generative AI / RAG?**

There are three main reasons why text splitting is a critical component when building LLM-powered applications, especially for RAG (Retrieval-Augmented Generation):

1.  **Overcomes Model Limitations:**
    *   Every LLM has a **context window limit** (a maximum input size, e.g., 50,000 tokens).
    *   If your document is larger than this limit (e.g., a 100,000-word PDF), you cannot feed it to the model in one go. Text splitting allows you to process the document in parts.

2.  **Improves Downstream Task Performance:**
    *   **For Embeddings:** Generating a single embedding vector for a very long text often fails to capture the semantic meaning accurately. Smaller, focused chunks produce better, more meaningful embeddings.
    *   **For Semantic Search:** Searching for information (like "Which team does Virat Kohli play for?") is more precise when you can compare the query embedding against embeddings of small, topic-specific chunks (e.g., separate chunks for RCB, CSK, MI) rather than one huge document embedding.
    *   **For Summarization:** LLMs tend to perform better and are less prone to hallucinating or drifting off-topic when summarizing smaller, coherent chunks of text rather than one enormous document.

3.  **Optimizes Computational Resources:**
    *   Processing smaller chunks requires less memory.
    *   It allows for better parallelization of tasks, making the overall process faster and more efficient.

#### **4. Summary & Recommendation**

| Splitter Type | Splitting Basis | Pros | Cons | Use Case |
| :--- | :--- | :--- | :--- | :--- |
| **Length-Based** | Character/Token Count | Very fast, simple | Ignores structure/meaning, can cut sentences/words | When speed is the only priority, or for very simple text. |
| **Text Structure-Based** | Paragraphs -> Sentences -> Words | **Most recommended.** Creates coherent chunks, respects text hierarchy | Slightly slower than length-based | **General-purpose text splitting for most RAG applications.** |
| **Document Structure-Based** | Language-specific syntax (e.g., `class`, `def`, headings) | Preserves logical blocks in code/markdown | Specific to document type | Splitting code files, Markdown docs, HTML. |
| **Semantic-Based** | Change in topic/meaning | Theoretically ideal for semantic coherence | Experimental, can be inaccurate, computationally heavy | Niche cases where topic segmentation is critical. |

**Final Recommendation:** For most projects, the **`RecursiveCharacterTextSplitter`** is your go-to tool. It provides the best balance of coherence, control, and performance.

In [3]:
import json
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load data
with open("./6_cleaned_socialinclusion.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Splitter (used only when needed)
splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=120,
    separators=["\n\n", "\n", ".", " ", ""],
)

documents = []

for i, item in enumerate(data):
    question = item["question"].strip()
    answer = item["answer"].strip()

    full_text = f"Question: {question}\n\nAnswer:\n{answer}"

    # ✅ Only split if large
    if len(full_text) > 1200:
        chunks = splitter.split_text(full_text)
    else:
        chunks = [full_text]

    # Store
    for j, chunk in enumerate(chunks):
        documents.append({
            "id": f"{i}_{j}",
            "text": chunk,
            "metadata": {
                "question": question,
                "type": "legal_qa",
                "chunk": j
            }
        })

print(f"Total chunks: {len(documents)}")

Total chunks: 2900


In [4]:
chunks[0]

'Question: I am a street child. The police regularly round me up, beat me, and take the money I\'ve earned from small jobs, calling me a \'thief\' and a \'nuisance,\' without any evidence of crime. Is this legal?\n\nAnswer:\n**No, this is torture, degrading treatment, robbery (or extortion), and unlawful confinement by public officials.**\n\n**Law says:**\n- National Penal (Code) Act, 2017 Section 167: "Prohibition of torture." Beating is physical torture.\n- Section 168: Degrading and inhuman treatment.\n- Section 244: "Prohibition of robbery." Taking money by force or show of force is robbery.\n- Section 200: Unlawful confinement.\n- Section 38: Aggravating factors apply, including abuse of public office.'

In [5]:
type(chunks[0])

str